In [1]:
import datetime
from concordia.agents import entity_agent
from concordia.agents import entity_agent_with_logging 
from concordia.associative_memory import basic_associative_memory
from concordia.components import agent as agent_components
from concordia.components.agent import concat_act_component
from concordia.contrib import language_models as language_model_utils
from concordia.contrib.language_models.ollama import ollama_model
from concordia.environment.engines import asynchronous
from concordia.prefabs.simulation import generic as simulation
import concordia.prefabs.game_master as game_master_prefabs
import concordia.prefabs.entity as entity_prefabs
from concordia.typing import prefab as prefab_lib
from concordia.utils import async_measurements as async_measurements_lib
from concordia.utils import helper_functions
import pandas as pd
from IPython import display
import numpy as np
from numpy.linalg import norm
import sentence_transformers
import ollama
import json
import os
import yaml
import random
import itertools
import time

In [2]:
os.environ["OLLAMA_HOST"] = "http://localhost:11434"

model = ollama_model.OllamaLanguageModel(
    model_name="deepseek-r1:8b",
)

In [3]:
with open('personas.yaml', 'r', encoding='utf-8') as f:
    yaml_data = yaml.safe_load(f)['personas']

In [4]:
class MpnetEmbedder:
    """Embedder using HuggingFace, adpted to Concordia."""
    
    def __init__(self, model):
        self._model = model
    def __call__(self, text: str) -> np.ndarray:
        return self._model.encode(text, show_progress_bar=False).astype(np.float32)
    def embed(self, text: str) -> np.ndarray:
        # Pega a string, encode, e garante que é um numpy array de float32
        return self._model.encode(text, show_progress_bar=False).astype(np.float32)

    def embed_sentence(self, text: str) -> np.ndarray:
        # Apenas um alias, pois algumas partes do Concordia preferem esse nome
        return self.embed(text)

In [5]:
st_model = sentence_transformers.SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
embedder = MpnetEmbedder(st_model)

In [7]:
def load_mapping():
    with open("mfq_mapping.json") as f:
        mapping = json.load(f)

    return mapping

In [8]:
def analyze_results(results, mapping):

    analysis = []

    for r in results:

        scores = compute_mfq_scores(
            r["answers"],
            mapping
        )

        analysis.append({
            "persona": r["persona"],
            **scores
        })

    return analysis

In [9]:
def compute_mfq_scores(answers, mapping):

    scores = {}

    for foundation, questions in mapping.items():

        values = [] 

        for q in questions:
            if q in answers:
                values.append(answers[q])

        scores[foundation] = np.mean(values)

    return scores

In [10]:
with open("results.json", "r", encoding="utf-8") as f:
    json_results = json.load(f)

mapping = load_mapping()

analysis = analyze_results(json_results, mapping)

df_mft = pd.DataFrame(analysis)

df_mft = df_mft.dropna(subset=['care', 'fairness', 'loyalty', 'authority', 'purity'])

In [11]:
def build_concordia_agent(persona_yaml, mft_scores, current_news):
    name = persona_yaml['name']
    
    interests = "\n- ".join(persona_yaml['interests'])
    beliefs = "\n- ".join(persona_yaml['beliefs'])
    
    # MUDANÇA 2: O Prompt agora força o agente a cuspir JSON purinho.
    identity_prompt = f"""
You are {name}, {persona_yaml['age_range']} years old, living in {persona_yaml['location']}.
Occupation: {persona_yaml['occupation']}
Description: {persona_yaml['small_description']}

Your main interests are:
- {interests}

Your core beliefs are:
- {beliefs}

YOUR MORAL FOUNDATIONS (Scale from 0 to 5, where 5 is the most important to you):
- Care/Harm: {mft_scores['care']:.2f}
- Fairness/Cheating: {mft_scores['fairness']:.2f}
- Loyalty/Betrayal: {mft_scores['loyalty']:.2f}
- Authority/Subversion: {mft_scores['authority']:.2f}
- Purity/Degradation: {mft_scores['purity']:.2f}

SOCIAL MEDIA BEHAVIOR INSTRUCTION:
You are participating in a highly engaging political and news discussion forum.
DO NOT post about your daily routine, or mundane personal life. 
CURRENT TRENDING TOPIC TO DISCUSS: "{current_news}"

CRITICAL RULES FOR INTERACTING - READ CAREFULLY:
1. Look at your observation feed. Real posts appear like this: [Post #0] "Title" by Author.
3. 1. FORBIDDEN ACTION: If you see ANY existing posts in your feed (e.g., [Post #0], [Post #1]), you are STRICTLY FORBIDDEN from using the "post" action to create a new thread. You MUST interact with the existing users.
4. MANDATORY ENGAGEMENT: You MUST use "reply", "upvote", or "downvote" on the existing posts. Look at the exact number in the feed and use it as the "post_id" (e.g., "post_id": "2").
5. If you want to reply to a post, you MUST use the EXACT number you see in the feed. For example, if you want to reply to [Post #0], you must output {{"action": "reply", "author": "{name}", "post_id": "0", "content": "..."}}.
6. NEVER invent, guess, or hallucinate a post_id (like "12345"). If you don't see any posts with IDs in your feed, DO NOT use the reply action.
7. If the forum is empty (you see "No new activity"), you MUST use the "post" action to share your strong opinion on the CURRENT TRENDING TOPIC.
8. HOW TO DEBATE: Evaluate the existing posts based on your Moral Foundations. 
   - If a user says something that attacks your high scores, you MUST use "reply" to argue with them, or use "downvote".
   - If a user says something you agree with, you MUST use "reply" to support them, or use "upvote".
9. If there are posts or replies from other users, read them. If they attack or support your moral foundations, "reply" to them directly or use "upvote" to approve and "downvote" to disapprove.
10. Do NOT talk about your daily routine, recipes, or random subjects. Focus strictly on debating the trending topic with other users.
11. Always give preference to replying to other users’ posts instead of creating new posts.

CRITICAL OUTPUT INSTRUCTION - YOU ARE AN API:
You are interacting directly with a digital forum backend. Every action you take MUST be a valid JSON object matching one of these exact formats. Do NOT output any markdown (like ```json), introduction, or conversational text outside the JSON.

Choose ONLY ONE action per turn from these options:

{{"action": "ignore", "author": "{name}"}}
{{"action": "reply", "author": "{name}", "post_id": "0", "content": "..."}}
{{"action": "upvote", "author": "{name}", "post_id": "0"}}
{{"action": "downvote", "author": "{name}", "post_id": "0"}}
{{"action": "post", "author": "{name}", "title": "...", "content": "..."}}

"""

    raw_memory_list = []
    memory = agent_components.memory.ListMemory(memory_bank=raw_memory_list)
    
    identity = agent_components.constant.Constant(state=identity_prompt)
    
    act_comp = concat_act_component.ConcatActComponent(model=model)
    
    agent = entity_agent_with_logging.EntityAgentWithLogging(
        agent_name=name,
        act_component=act_comp, 
        context_components={    
            'memory': memory,
            'identity': identity
        },
        measurements=async_measurements_lib.ReactiveMeasurements()
    )
    return agent

In [15]:
# ==========================================
# VARIÁVEIS DE CONTROLE DO EXPERIMENTO
# ==========================================
num_batches = 10    # Quantos fóruns independentes você quer rodar no total
batch_size = 5      # Quantas pessoas (agentes) por fórum

print(f"\nIniciando bateria de {num_batches} experimentos aleatórios...")

news_list = [
    """SYSTEM BROADCAST POST: Children’s Hospital Receives Record Donation and Expands Treatment for Rare Diseases

A nationwide campaign raised millions for a children’s hospital, allowing the opening of new wings and free treatment for hundreds of vulnerable children.
""",

    """SYSTEM BROADCAST POST: Congressman Uses Public Funds for Personal Trip and Faces Investigation

Documents indicate that resources allocated to the congressional office were used for personal expenses abroad.
""",

    """SYSTEM BROADCAST POST: School Reintroduces Daily National Anthem and Stricter Discipline Rules

The administration states that the measure aims to reinforce respect, order, and civic values among students.
""",

    """SYSTEM BROADCAST POST: Religious Leader Criticizes School Campaign Supporting Same-Sex Marriage

During a public event, a religious leader stated that schools should prioritize academic content and preserve traditional family values.
""",

    """SYSTEM BROADCAST POST: Historic Drought Forces City to Ration Water and Suspend Classes

Reservoirs reached critical levels after months without rain. Experts linked the extreme conditions to the advance of climate change.
""",

    """SYSTEM BROADCAST POST: Congress Approves Bill Tightening Rules for Gun Purchase and Ownership

Lawmakers approved new requirements for firearms acquisition, including training, psychological evaluations, and tracking measures. Supporters argue the policy could reduce deaths and accidents.
"""
]

base_dir = r"D:\breno\Documents\TCC\Resultados"


os.makedirs(base_dir, exist_ok=True)

current_folders = [f for f in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, f)) and f.startswith("Experimento ")]

next_num = 8 
if current_folders:
    numbers = []
    for folder in current_folders:
        try:
            num = int(folder.replace("Experimento ", ""))
            numbers.append(num)
        except ValueError:
            pass
    if numbers:
        next_num = max(max(numbers) + 1, 8)

new_folder = os.path.join(base_dir, f"Experimento {next_num}")
os.makedirs(new_folder, exist_ok=True)
print(f"\n📂 Diretório criado para salvar resultados: {new_folder}")

persona_list = []
full_start_time = time.time()
# --- COMEÇO DO NOVO LOOP ---
for batch_index in range(num_batches):
    print(f"\n{'='*60}")
    print(f" EXPERIMENTO {batch_index + 1} DE {num_batches} ")
    print(f"{'='*60}")
    batch_start_time = time.time()

    current_news = news_list[batch_index % len(news_list)]

    first_post = f"Event: System_Admin created post #999: 'BREAKING NEWS'\n{current_news}"
    print(f"\n[TEMA DO FÓRUM]: {current_news}")

    if len(persona_list) < batch_size:
        print("\n[SISTEMA] Embaralhando as 40 personas para garantir participação de todos...")
        persona_list = list(yaml_data) # Pega uma cópia da lista completa
        random.shuffle(persona_list)   # Embaralha tudo

    # "Compre" as 5 primeiras cartas do baralho
    batch_group = persona_list[:batch_size]
    
    # Remova essas 5 cartas do baralho, para que não se repitam nos próximos lotes
    persona_list = persona_list[batch_size:]
    
    active_agents = []
    vetores_mft_do_grupo = [] # Lista que vai guardar as notas para a matemática de similaridade
    
    print("\n--- COMPOSIÇÃO DO GRUPO ---")
    
    active_agents = []
    vetores_mft_do_grupo = [] # Vai guardar as listas [care, fairness, ...]
    
    for persona_yaml in batch_group:
        name = persona_yaml['name']
        mft_row = df_mft[df_mft['persona'] == name]
        
        if not mft_row.empty:
            mft_scores = mft_row.iloc[0]
            
            # Extrai o VETOR da persona: [care, fairness, loyalty, authority, purity]
            vetor_mft = mft_scores[['care', 'fairness', 'loyalty', 'authority', 'purity']].astype(float).values
            vetores_mft_do_grupo.append(vetor_mft)
            
            # Printa o vetor individual para visualização
            print(f" -> {name:<25} | Vetor: [{', '.join([f'{v:.1f}' for v in vetor_mft])}]")
            
            # Cria o agente no Concordia
            agent = build_concordia_agent(persona_yaml, mft_scores, current_news)
            active_agents.append(agent)
        else:
            print(f" -> AVISO: Dados de MFT não encontrados para {name}")

    # --- MATEMÁTICA VETORIAL DO GRUPO ---
    if len(vetores_mft_do_grupo) > 1:
        matriz_mft = np.array(vetores_mft_do_grupo)
        
        # 1. Média do grupo PARA CADA fundamento
        medias_grupo = np.mean(matriz_mft, axis=0)
        print("\n--- MÉDIA MORAL DO GRUPO (PERFIL DO FÓRUM) ---")
        print(f"Care: {medias_grupo[0]:.2f} | Fairness: {medias_grupo[1]:.2f} | "
              f"Loyalty: {medias_grupo[2]:.2f} | Authority: {medias_grupo[3]:.2f} | Purity: {medias_grupo[4]:.2f}")

        # 2. Grau de Similaridade Vetorial Média (Cosine Similarity entre todos os pares)
        similaridades = []
        for vec1, vec2 in itertools.combinations(matriz_mft, 2):
            norm_v1, norm_v2 = norm(vec1), norm(vec2)
            # Prevenção de divisão por zero
            if norm_v1 > 0 and norm_v2 > 0: 
                # Fórmula Cosine Sim: (A . B) / (||A|| * ||B||)
                cos_sim = np.dot(vec1, vec2) / (norm_v1 * norm_v2)
                similaridades.append(cos_sim)

        if similaridades:
            # Como as notas MFT são de 0 a 5 (todas positivas), a similaridade já sai entre 0 (0%) e 1 (100%)
            porcentagem_sim = np.mean(similaridades) * 100
            
            print(f"\n[MÉTRICA] Similaridade Vetorial do Grupo: {porcentagem_sim:.1f}%")
            
            # Nota estatística: Na similaridade de cosseno com notas positivas, 
            # os valores costumam ficar naturalmente altos.
            if porcentagem_sim > 93:
                print("          (Forte Bolha Ideológica / Vetores muito alinhados)")
            elif porcentagem_sim < 82:
                print("          (Fórum Polarizado / Diversidade Vetorial Alta)")

    print("\n--- INICIANDO FÓRUM ASSÍNCRONO ---")

    # --- O RESTO DO SEU CÓDIGO CONTINUA IGUAL AQUI ---
    gm_memory_bank = basic_associative_memory.AssociativeMemoryBank(
        sentence_embedder=embedder
    )

    gm_prefab = game_master_prefabs.async_social_media.GameMaster( 
        entities=active_agents,
        params={
            'name': 'Rural News',
            'forum_name': f'Rural News(Exp {batch_index + 1})', # Nome atualizado com o nº do exp
            # 'forum_name': f'Voz Progressista News(Exp {batch_index + 1})', # Nome atualizado com o nº do exp
            # 'forum_name': f'Voz Conservadora News(Exp {batch_index + 1})', # Nome atualizado com o nº do exp

            'measurements': async_measurements_lib.ReactiveMeasurements()
        }
    )

    gm = gm_prefab.build(model=model, memory_bank=gm_memory_bank)

    sim_engine = asynchronous.Asynchronous()
    
    entities = active_agents + [gm]
    
    # Dá a largada na simulação usando o run_loop!
    sim_engine.run_loop(
        game_masters=[gm],           
        entities=active_agents,      
        premise=first_post,           
        max_steps=4,                 
        verbose=True                 
    )
    
    print(f"\n--- FIM DO EXPERIMENTO {batch_index + 1} ---")
    
    # Imprime o feed final
    file_name = f"output-run-{batch_index + 1}.txt"
    file_path = os.path.join(new_folder, file_name)
    
    # Abre o arquivo com UTF-8 para não dar erro nos acentos do português
    with open(file_path, "w", encoding="utf-8") as txt_file:
        
        # Opcional, mas muito bom pro TCC: Salva o cabeçalho no TXT
        txt_file.write(f"TEMA: {current_news}\n")
        if similaridades:
            txt_file.write(f"SIMILARIDADE DO GRUPO: {porcentagem_sim:.1f}%\n")
        txt_file.write(f"{'='*50}\nFEED DA SIMULAÇÃO:\n{'='*50}\n\n")

        # Puxamos os últimos 30 logs para garantir que pega a conversa inteira
        recent_logs = gm.get_component('__memory__').retrieve_recent(limit=30) 
        
        for log in recent_logs:
            if "action" in log.lower() or "post" in log.lower() or "reply" in log.lower():
                # Printa no terminal para você acompanhar
                print(f"[FEED FINAL] {log}")
                
                # Escreve no arquivo de texto
                txt_file.write(f"{log}\n\n")
                
    print(f"💾 Log salvo em: {file_path}")

    batch_end_time = time.time()
    batch_time = batch_end_time - batch_start_time
    min_b, sec_b = divmod(batch_time, 60)
    print(f"\n[TEMPO] ⏱️ O Experimento {batch_index + 1} levou {int(min_b)}m {int(sec_b)}s para rodar.")

    full_end_time = time.time()
    fulltime = full_end_time - full_start_time
    min_t, sec_t = divmod(fulltime, 60)
    
    print(f"\n{'='*60}")
    print(f" SIMULAÇÃO COMPLETA FINALIZADA! 🎉")
    print(f" ⏱️ Tempo total de execução: {int(min_t)} minutos e {int(sec_t)} segundos.")
    print(f"{'='*60}")


Iniciando bateria de 10 experimentos aleatórios...

📂 Diretório criado para salvar resultados: D:\breno\Documents\TCC\Resultados\Experimento 23

 EXPERIMENTO 1 DE 10 

[TEMA DO FÓRUM]: SYSTEM BROADCAST POST: Children’s Hospital Receives Record Donation and Expands Treatment for Rare Diseases

A nationwide campaign raised millions for a children’s hospital, allowing the opening of new wings and free treatment for hundreds of vulnerable children.


[SISTEMA] Embaralhando as 40 personas para garantir participação de todos...

--- COMPOSIÇÃO DO GRUPO ---
 -> Rafael Andrade            | Vetor: [4.0, 3.7, 2.2, 2.0, 1.8]
 -> Adriana Souza             | Vetor: [4.2, 4.0, 2.5, 2.0, 1.8]
 -> Lucas Pereira             | Vetor: [4.5, 4.2, 2.2, 1.8, 1.8]
 -> André Luiz Barbosa        | Vetor: [3.5, 4.3, 2.2, 2.2, 1.6]
 -> Diego Santos              | Vetor: [4.5, 4.3, 3.8, 2.5, 2.4]

--- MÉDIA MORAL DO GRUPO (PERFIL DO FÓRUM) ---
Care: 4.13 | Fairness: 4.10 | Loyalty: 2.60 | Authority: 2.10 | Purit

In [13]:
instances = [
    prefab_lib.InstanceConfig(
        prefab='basic__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': 'Oliver Cromwell',
            'goal': 'become lord protector',
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='basic__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': 'King Charles I',
            'goal': 'avoid execution for treason',
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='generic__GameMaster',
        role=prefab_lib.Role.GAME_MASTER,
        params={
            'name': 'default rules',
            # Comma-separated list of thought chain steps.
            'extra_event_resolution_steps': '',
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='formative_memories_initializer__GameMaster',
        role=prefab_lib.Role.INITIALIZER,
        params={
            'name': 'initial setup rules',
            'next_game_master_name': 'default rules',
            'shared_memories': [
                'The king was captured by Parliamentary forces in 1646.',
                'Charles I was tried for treason and found guilty.',
            ],
        },
    ),
]

In [14]:
import inspect
assinatura = inspect.signature(asynchronous.Asynchronous.__init__)
print(f"O Motor Assíncrono exige estes parâmetros: {assinatura}")

O Motor Assíncrono exige estes parâmetros: (self, call_to_make_observation: str = 'What is the current situation faced by {name}? What do they now observe? Only include information of which they are aware.', call_to_next_acting: str = 'Which entities act next?', call_to_next_action_spec: str = 'In what action spec format should {name} respond? Respond in  one of the provided formats and use no additional words.', call_to_resolve: str = 'Because of all that came before, what happens next?', call_to_check_termination: str = 'Is the game/simulation finished?', call_to_next_game_master: str = 'Which rule set should we use for the next step?', sleep_time: float = 0.1)
